# 第52章 热力图（heatmap）

用颜色矩阵展示相关系数、透视表或任意二维数值。

## 学习目标

本章围绕一种明确的图表结构展开，先看最小可用示例，再加入分组、注释或交互细节。学习重点不是“把图画出来”，而是让图表服务于一个可回答的问题。


## 适用场景

比较行×列组合或压缩读取数值矩阵。

## 数据结构

二维矩阵或可透视成长×宽矩阵的长表。

## 本章练习任务

运行基础图表后，完成以下任务：

1. 将 cmap="vlag" 改为 cmap="coolwarm" 或 "RdBu_r"，对比不同发散色盘的视觉效果
2. 修改 center=0 为不设置 center，观察色阶中心对相关矩阵显示的影响
3. 调整 fmt=".2f" 为 fmt=".0f"，说明标注精度对数值可读性的作用


## 0. 准备可复现数据

先完成导入和数据准备，后续单元格只负责一种图表或一种分析动作。


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns


sns.set_theme(style="whitegrid", context="notebook")
diamonds = pd.read_csv(f"{base_url}/datasets/diamonds.csv")
orders_full = diamonds.assign(
    category = diamonds["cut"], channel=diamonds["color"], region=diamonds["clarity"],
    order_value = diamonds["price"], items=diamonds["carat"],
    satisfied=np.where(diamonds["price"] >= diamonds["price"].median(), "高于中位价", "不高于中位价"),
)
orders = orders_full.sample(2_000, random_state=36).copy()
taxis = pd.read_csv(f"{base_url}/datasets/taxis.csv", parse_dates=["pickup", "dropoff"])
marketing_full = taxis.assign(
    channel = taxis["payment"].fillna("unknown"), visits=taxis["distance"],
    ad_spend = taxis["tip"], sales=taxis["total"],
    conversion = (taxis["tip"] / taxis["total"].replace(0, np.nan)).fillna(0),
)
marketing = marketing_full.sample(min(2_000, len(marketing_full)), random_state=36).copy()
flights = pd.read_csv(f"{base_url}/datasets/flights.csv")
daily = flights.assign(
    date = pd.to_datetime(flights["year"].astype("string") + "-" + flights["month"] + "-01"),
    region = "AirPassengers", sales=flights["passengers"],
)
print(f"Diamonds：{len(diamonds):,} 行；NYC Taxis：{len(taxis):,} 行；Flights：{len(flights):,} 行")
print("图表兼容列均由公开数据原始字段直接映射；高成本图使用固定 2,000 行样本")


## 1. 基础图表

先保留必要的编码：位置、颜色或大小。图表标题、坐标轴和单位应能让读者脱离代码理解结果。


In [ ]:
corr = marketing[["visits", "ad_spend", "sales", "conversion"]].corr()
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="vlag", center=0, vmin=-1, vmax=1, square=True, ax=ax)
ax.set_title("营销指标相关系数")
fig.tight_layout()
plt.show()


## 2. 进阶变体

在基础图表可读的前提下增加分组、布局、注释或交互。新增编码必须服务于一个明确问题。


In [ ]:
pivot = orders.pivot_table(index="region", columns="category", values="order_value", aggfunc="mean")
fig, ax = plt.subplots(figsize=(7.5, 4))
sns.heatmap(pivot, annot=True, fmt=".0f", cmap="Blues", linewidths=0.5, cbar_kws={"label": "平均客单价（元）"}, ax=ax)
ax.set(title="区域与品类客单价", xlabel="品类", ylabel="区域")
fig.tight_layout()
plt.show()


## 3. 参数说明

- annot：标注
- fmt：格式
- cmap：色盘
- center/vmin/vmax：色阶


## 4. 结果解读

先读色阶含义，再找极值、带状结构和异常组合。


## 常见误区

- 色阶范围随数据变化无法跨图比较
- 相关矩阵使用单向色盘
- 标注过密


## 综合练习

请使用同一份数据完成下面任务，并说明你选择该图表的原因。完成后补充：图表回答了什么问题、最重要的视觉信号是什么、还有哪些信息无法从图中得出。


In [ ]:
counts = pd.crosstab(orders["region"], orders["channel"])
fig, ax = plt.subplots(figsize=(7.5, 4))
sns.heatmap(counts, annot=True, fmt="d", cmap="YlGnBu", linewidths=0.5, ax=ax)
ax.set(title="区域与渠道订单量", xlabel="渠道", ylabel="区域")
fig.tight_layout()
plt.show()


## 本章小结

用颜色矩阵展示相关系数、透视表或任意二维数值。


### 你已经掌握

- 判断热力图（heatmap）的适用场景
- 准备与图表匹配的数据结构
- 从基础图表扩展到分组、注释或交互变体
- 按照业务问题解读图表并说明结论边界


### 图表选择速查

| 选择要点 | 本章说明 |
| --- | --- |
| 适用场景 | 比较行×列组合或压缩读取数值矩阵。 |
| 数据结构 | 二维矩阵或可透视成长×宽矩阵的长表。 |
| 结果解读 | 先读色阶含义，再找极值、带状结构和异常组合。 |


### 关键参数

| 参数 | 作用 |
| --- | --- |
| `annot` | 标注 |
| `fmt` | 格式 |
| `cmap` | 色盘 |
| `center/vmin/vmax` | 色阶 |


### 需要注意

- 色阶范围随数据变化无法跨图比较
- 相关矩阵使用单向色盘
- 标注过密


### 完成检查

- [ ] 能判断什么问题适合使用热力图（heatmap）
- [ ] 能准备符合要求的数据结构
- [ ] 能独立完成基础图表和一个进阶变体
- [ ] 能调整关键参数并解释视觉变化
- [ ] 能根据图表写出有边界的数据结论


### 下一步推荐

把同一图表迁移到另一份数据，先保留同样的编码，再只改变一个维度。比较迁移前后的可读性，并说明哪些结论仍然成立。
